# **GroupBy Operations & Multi-Level Index in Pandas**

A groupby operation allows us to **analyze data per category.**

In plain English, it helps us answer questions like:

- What is the **average value per group**?
- How many observations exist **per category**?
- How does a numerical feature behave **within each group**?

#### **The Core Mental Model**

Think of groupby as a **split → apply → combine** process:

1. **Split** the data based on categories
2. **Apply** an aggregation function
3. **Combine** the results into a new DataFrame

pandas-groupby-split-apply-combine.svg

<br>

## **Understanding the Logic**

Imagine we have the following table of students and their exam scores:

| Student | Class   | Score |
| ------- | ------- | ----- |
| Alex    | Math    | 85    |
| Jamie   | Math    | 90    |
| Sam     | Physics | 78    |
| Taylor  | Physics | 82    |
| Jordan  | CS      | 95    |
| Casey   | CS      | 88    |



### **Identify the Roles of Each Column**

- **Class** → categorical variable
- **Score** → continuous numerical variable

Why?

- A **categorical variable** defines **groups**
  - Math, Physics, CS
- A **continuous variable** is something we want to summarize
  - Scores can be averaged, summed, counted, etc.

<br>

💡***Important concept***

A categorical variable does **not** need to be text.
Numbers like 1, 2, 3 (e.g., grade level, model year, cabin class) can also define categories.




## **What “Grouping” Actually Means**

#### **Step 1: Split the Table by Category**

If we group by Class, the table is mentally separated into smaller tables:

**Math**

| Student | Score |
| ------- | ----- |
| Alex    | 85    |
| Jamie   | 90    |

<br>

**Physics**
| Student | Score |
| ------- | ----- |
| Sam     | 78    |
| Taylor  | 82    |

<br>

**CS**
| Student | Score |
| ------- | ----- |
| Jordan  | 95    |
| Casey   | 88    |


<br>

At this stage:

- **No math operation has happened**
- We have only **organized the data**

This is a crucial idea.


## **Grouping Alone Does Nothing (Lazy Evaluation)**


Just **grouping** the data does not produce numbers yet.

Why?

Because the computer still doesn’t know:

- Do you want the average score?
- The highest score?
- The number of students?

So grouping by itself means:

*“I know how to split the data — now tell me what to compute.”*

This is what people mean when they say **groupby is lazy**.

#### **Step 2: Apply an Aggregation**

Now we tell the computer **what calculation** to perform.

**Example A: Average Score per Class**

We compute the mean within each group:

- Math → (85 + 90) / 2 = 87.5
- Physics → (78 + 82) / 2 = 80
- CS → (95 + 88) / 2 = 91.5

Resulting summary table:

| Class   | Average Score |
| ------- | ------------- |
| Math    | 87.5          |
| Physics | 80            |
| CS      | 91.5          |

<br>

**Example B: Count per Class**

Instead of averaging, we could count rows:

- Math → 2 students
- Physics → 2 students
- CS → 2 students


| Class   | Number of Students |
| ------- | ------------------ |
| Math    | 2                  |
| Physics | 2                  |
| CS      | 2                  |

<br>

⚠️ Key distinction:

- Count → counts rows
- Sum → adds numerical values

#### **Step 3: Combine the Results**

Finally, the individual group results are combined into a new table:

- One row **per category**
- One column **per aggregation result**

This final table is what groupby ultimately returns.

## **Big Picture Takeaway**

Before writing any code, you should always be able to answer:

1. What is my categorical variable?
2. What numerical column am I summarizing?
3. What question am I asking per group?

If you can answer those three questions, groupby becomes straightforward — pandas is just the implementation.

## **GroupBy in Pandas**

We’ll work with the MPG dataset, which contains information about car models, including:

- fuel efficiency (mpg)
- country of origin (origin)
- engine characteristics
- model year

#### **Step 1: Load and Inspect the Data**

In [ ]:
import pandas as pd

df = pd.read_csv("mpg.csv")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


The MPG dataset contains information about various car models. Key columns include:

- **mpg**: Fuel efficiency (miles per gallon), a continuous numerical variable.
- **cylinders**: Number of cylinders in the engine, a categorical variable.
- **displacement**: Engine displacement, a continuous numerical variable.
- **horsepower**: Engine horsepower, a continuous numerical variable.
- **weight**: Vehicle weight, a continuous numerical variable.
- **acceleration**: Time to accelerate from 0 to 60 mph, a continuous numerical variable.
- **model_year**: Year the car model was released, a categorical variable.
- **origin**: Country of origin (e.g., 1 for US, 2 for Europe, 3 for Japan), a categorical variable.
- **name**: Name of the car model, a categorical variable.

#### **Step2: Identify Categorical vs Continuous Columns, Based on the Question**

How does fuel efficiency differ by car origin?

This is a very natural groupby question because:

- origin defines **distinct categories**
- mpg is a **numerical variable** we want to summarize

In [ ]:
df['origin'].value_counts()

,count
origin,
1,249
3,79
2,70


⚠️ **Reminder:**

Even though origin is numeric, it still represents categories, not quantities.

#### **Step 3: Create a GroupBy Object (No Computation Yet)**

In [ ]:
df.groupby('origin')

This returns a **GroupBy object**, not results.

**What this means conceptually:**

- Pandas has split the data into groups
- No math has been done yet
- Pandas is waiting for instructions

📌 This is **lazy evaluation** again — same idea as in the conceptual example.

#### **Step 4: Apply an Aggregation Function**

Now we tell pandas **what to compute per group.**

Example: Average MPG per Origin

In [ ]:
# This will generate ERRROR, can you guess why?

df.groupby('origin').mean()

TypeError: agg function failed [how->mean,dtype->object]

Pandas does the following:

1. Groups rows by origin
2. Tries to compute the mean of every column
3. Skips non-numeric columns only if it can clearly identify them as non-numeric.
4. ❌ Fails if a column looks numeric but is actually stored as strings

**Identify the Problematic Columns**

So, Always inspect dtypes before aggregating:

In [ ]:
df.dtypes

,0
mpg,float64
cylinders,int64
displacement,float64
horsepower,object
weight,int64
acceleration,float64
model_year,int64
origin,int64
name,object


🚨 Red flag:

Any numeric-looking column stored as object is dangerous for aggregation.

#### **Decide What You Actually Want to Average**

Pedagogically, this is an important lesson:

 - Groupby does not mean “average everything.”
 - It means “average the columns that make sense.”


**Option A: Select Columns Explicitly**

If your question is: *What is the average MPG per origin?*

Then be explicit:

In [ ]:
df.groupby('origin')['mpg'].mean()

,mpg
origin,
1,20.083534
2,27.891429
3,30.450633


**Option B: Choose Numeric Only Columns**

In [ ]:
df.groupby('origin').mean(numeric_only= True)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
origin,,,,,,,
1,20.083534,6.248996,245.901606,119.048980,3361.931727,15.033735,75.610442
2,27.891429,4.157143,109.142857,80.558824,2423.300000,16.787143,75.814286
3,30.450633,4.101266,102.708861,79.835443,2221.227848,16.172152,77.443038


**Option C: Fix Data Types Properly**

If you do want to aggregate multiple numeric columns, you should clean the data first.

Example: Fix horsepower

In [ ]:
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

What this does:

- Converts valid numbers to numeric
-  `errors='coerce'`: any values that cannot be converted to a number (e.g., strings like '?', or other non-numeric text) will be replaced with NaN

Now pandas can safely compute means.

Retry the GroupBy, it still might not work, so perhaps stick with Option A

In [ ]:
df.groupby('origin').mean()


TypeError: agg function failed [how->mean,dtype->object]

#### **Step 5: Understanding the Output**

Look closely at the result:

- origin is now the index, not a column


In [ ]:
df.groupby('origin')['mpg'].mean()

,mpg
origin,
1,20.083534
2,27.891429
3,30.450633


<br>

Common aggregation function options:

    mean(): Compute mean of groups
    sum(): Compute sum of group values
    size(): Compute group sizes
    count(): Compute count of group
    std(): Standard deviation of groups
    var(): Compute variance of groups
    sem(): Standard error of the mean of groups
    describe(): Generates descriptive statistics
    first(): Compute first of group values
    last(): Compute last of group values
    nth() : Take nth value, or a subset if n is a list
    min(): Compute min of group values
    max(): Compute max of group values
    
Full List at the Online Documentation: https://pandas.pydata.org/docs/reference/groupby.html

<br><br>

## **Using describe() with GroupBy**

In [ ]:
df.groupby('origin').describe()

mpg                                                     cylinders  \
        count       mean       std   min   25%   50%    75%   max     count   
origin                                                                        
1       249.0  20.083534  6.402892   9.0  15.0  18.5  24.00  39.0     249.0   
2        70.0  27.891429  6.723930  16.2  24.0  26.5  30.65  44.3      70.0   
3        79.0  30.450633  6.090048  18.0  25.7  31.6  34.05  46.6      79.0   

                  ... acceleration       model_year                       \
            mean  ...          75%   max      count       mean       std   
origin            ...                                                      
1       6.248996  ...        16.90  22.2      249.0  75.610442  3.677094   
2       4.157143  ...        18.90  24.8       70.0  75.814286  3.469506   
3       4.101266  ...        17.55  21.0       79.0  77.443038  3.650595   

                                      
         min   25%   50%   75%   max  
origin                                
1       70.0  73.0  76.0  79.0  82.0  
2       70.0  73.0  76.0  79.0  82.0  
3       70.0  74.0  78.0  81.0  82.0  

[3 rows x 56 columns]

In [ ]:
df.groupby('model_year').describe().transpose()

model_year                   70           71           72           73  \
mpg          count    29.000000    28.000000    28.000000    40.000000   
             mean     17.689655    21.250000    18.714286    17.100000   
             std       5.339231     6.591942     5.435529     4.700245   
             min       9.000000    12.000000    11.000000    11.000000   
             25%      14.000000    15.500000    13.750000    13.000000   
             50%      16.000000    19.000000    18.500000    16.000000   
             75%      22.000000    27.000000    23.000000    20.000000   
             max      27.000000    35.000000    28.000000    29.000000   
cylinders    count    29.000000    28.000000    28.000000    40.000000   
             mean      6.758621     5.571429     5.821429     6.375000   
             std       1.724926     1.665079     2.073708     1.807215   
             min       4.000000     4.000000     3.000000     3.000000   
             25%       6.000000     4.000000     4.000000     4.000000   
             50%       8.000000     6.000000     4.000000     7.000000   
             75%       8.000000     6.500000     8.000000     8.000000   
             max       8.000000     8.000000     8.000000     8.000000   
displacement count    29.000000    28.000000    28.000000    40.000000   
             mean    281.413793   209.750000   218.375000   256.875000   
             std     124.421380   115.102410   123.781964   121.722085   
             min      97.000000    71.000000    70.000000    68.000000   
             25%     198.000000    97.750000   109.250000   121.750000   
             50%     307.000000   228.500000   131.000000   276.000000   
             75%     383.000000   273.000000   326.000000   350.250000   
             max     455.000000   400.000000   429.000000   455.000000   
horsepower   count    29.000000    27.000000    28.000000    40.000000   
             mean    147.827586   107.037037   120.178571   130.475000   
             std      53.734844    38.566109    41.121368    46.412304   
             min      46.000000    60.000000    54.000000    46.000000   
             25%      95.000000    81.000000    86.750000    93.250000   
             50%     150.000000    95.000000   104.500000   129.500000   
             75%     198.000000   130.000000   150.750000   160.250000   
             max     225.000000   180.000000   208.000000   230.000000   
weight       count    29.000000    28.000000    28.000000    40.000000   
             mean   3372.793103  2995.428571  3237.714286  3419.025000   
             std     852.868663  1061.830859   974.520960   974.809133   
             min    1835.000000  1613.000000  2100.000000  1867.000000   
             25%    2648.000000  2110.750000  2285.500000  2554.500000   
             50%    3449.000000  2798.000000  2956.000000  3338.500000   
             75%    4312.000000  3603.250000  4169.750000  4247.250000   
             max    4732.000000  5140.000000  4633.000000  4997.000000   
acceleration count    29.000000    28.000000    28.000000    40.000000   
             mean     12.948276    15.142857    15.125000    14.312500   
             std       3.330982     2.666171     2.850032     2.754222   
             min       8.000000    11.500000    11.000000     9.500000   
             25%      10.000000    13.375000    13.375000    12.500000   
             50%      12.500000    14.500000    14.500000    14.000000   
             75%      15.000000    16.125000    16.625000    16.000000   
             max      20.500000    20.500000    23.500000    21.000000   
origin       count    29.000000    28.000000    28.000000    40.000000   
             mean      1.310345     1.428571     1.535714     1.375000   
             std       0.603765     0.741798     0.792658     0.667467   
             min       1.000000     1.000000     1.000000     1.000000   
             25%       1.000000     1.000000     1.000000     1.000000   
    

<br><br><br>

### **Practice Problems**

**Question:** What is the average fuel efficiency (miles per gallon, `mpg`) for cars with different numbers of cylinders?


In [ ]:
# Solution:

,mpg
cylinders,
3,20.550000
4,29.286765
5,27.366667
6,19.985714
8,14.963107


<br>

**Question:** What is the average `acceleration` for cars produced in each `model_year`?



In [ ]:
# Solution:

,acceleration
model_year,
70,12.948276
71,15.142857
72,15.125000
73,14.312500
74,16.203704
75,16.050000
76,15.941176
77,15.435714
78,15.805556


<br>

**Question:** What is the total `weight` of cars from each `origin`?


In [ ]:
# Solution:

,weight
origin,
1,837121
2,169631
3,175477


<br>

**Question:** How many cars were produced in each `model_year`?


In [22]:
# Solution:

,name
model_year,
70,29
71,28
72,28
73,40
74,27
75,30
76,34
77,28
78,36


<br><br>

# **Grouping by Multiple Columns & Advanced Aggregation**

## **Multi-Level Grouping (MultiIndex)**


We’ve seen average MPG by origin. Now let’s explore a deeper question:

How does MPG vary not just by origin, but also by the number of cylinders in the car?

In [ ]:
# Group by origin and cylinders, compute mean for numeric columns
origin_cyl = df.groupby(['origin', 'cylinders']).mean(numeric_only=True)
origin_cyl


mpg  displacement  horsepower       weight  \
origin cylinders                                                     
1      4          27.840278    124.284722   80.956522  2437.166667   
       6          19.663514    226.283784   99.671233  3213.905405   
       8          14.963107    345.009709  158.300971  4114.718447   
2      4          28.411111    104.222222   78.311475  2330.015873   
       5          27.366667    145.000000   82.333333  3103.333333   
       6          20.100000    159.750000  113.500000  3382.500000   
3      3          20.550000     72.500000   99.250000  2398.500000   
       4          31.595652     99.768116   75.579710  2153.492754   
       6          23.883333    156.666667  115.833333  2882.000000   

                  acceleration  model_year  
origin cylinders                            
1      4             16.526389   78.027778  
       6             16.474324   75.635135  
       8             12.955340   73.902913  
2      4             16.722222   75.507937  
       5             18.633333   79.000000  
       6             16.425000   78.250000  
3      3             13.250000   75.500000  
       4             16.569565   77.507246  
       6             13.550000   78.000000

**Observation:**

- The resulting DataFrame has a MultiIndex hierarchy
  - Level 0 → origin
  - Level 1 → cylinders
- Only numeric columns are aggregated
- Non-numeric columns (e.g., name) are automatically dropped

### **Inspecting the MultiIndex**

How can we explore the structure of this hierarchical index?

In [ ]:
origin_cyl.index

MultiIndex([(1, 4),
            (1, 6),
            (1, 8),
            (2, 4),
            (2, 5),
            (2, 6),
            (3, 3),
            (3, 4),
            (3, 6)],
           names=['origin', 'cylinders'])

origin_cyl.index.names shows the names of each level

In [ ]:
origin_cyl.index.names

FrozenList(['origin', 'cylinders'])

origin_cyl.index.levels shows the unique values in each level

In [ ]:
origin_cyl.index.levels

FrozenList([[1, 2, 3], [3, 4, 5, 6, 8]])

<br>

### **Selecting Data from a MultiIndex**

In [ ]:
origin_cyl = df.groupby(['origin', 'cylinders']).mean(numeric_only=True)
origin_cyl

mpg  displacement  horsepower       weight  \
origin cylinders                                                     
1      4          27.840278    124.284722   80.956522  2437.166667   
       6          19.663514    226.283784   99.671233  3213.905405   
       8          14.963107    345.009709  158.300971  4114.718447   
2      4          28.411111    104.222222   78.311475  2330.015873   
       5          27.366667    145.000000   82.333333  3103.333333   
       6          20.100000    159.750000  113.500000  3382.500000   
3      3          20.550000     72.500000   99.250000  2398.500000   
       4          31.595652     99.768116   75.579710  2153.492754   
       6          23.883333    156.666667  115.833333  2882.000000   

                  acceleration  model_year  
origin cylinders                            
1      4             16.526389   78.027778  
       6             16.474324   75.635135  
       8             12.955340   73.902913  
2      4             16.722222   75.507937  
       5             18.633333   79.000000  
       6             16.425000   78.250000  
3      3             13.250000   75.500000  
       4             16.569565   77.507246  
       6             13.550000   78.000000

How can we select all cars from origin 1?

In [ ]:
# All cars from origin 1
origin_cyl.loc[1]

,mpg,displacement,horsepower,weight,acceleration,model_year
cylinders,,,,,,
4,27.840278,124.284722,80.956522,2437.166667,16.526389,78.027778
6,19.663514,226.283784,99.671233,3213.905405,16.474324,75.635135
8,14.963107,345.009709,158.300971,4114.718447,12.955340,73.902913


How can we select cars with 6 cylinders across all origins?

In [ ]:
# All 6-cylinder cars across all origins
origin_cyl.xs(key=6, level='cylinders')

,mpg,displacement,horsepower,weight,acceleration,model_year
origin,,,,,,
1,19.663514,226.283784,99.671233,3213.905405,16.474324,75.635135
2,20.100000,159.750000,113.500000,3382.500000,16.425000,78.250000
3,23.883333,156.666667,115.833333,2882.000000,13.550000,78.000000


- .loc[] selects along the outer index
- .xs() slices a specific level

<br>

Since after selection, the cylinders level is dropped from the MultiIndex in the resulting DataFrame and it is not obvious they are all 6-cylinder cars we might want to assign an informative name for that

In [ ]:
all_6_cylinder =origin_cyl.xs(key=6, level='cylinders')

<br>

What if we want to select multiple keys, for example 6 and 8? While it's possible to do this with `.xs()` by passing a list of keys (e.g., `origin_cyl.xs(key=[6, 8], level='cylinders')`), it's often more intuitive and sometimes easier to filter out those values from the original DataFrame *before* you run the `groupby` call, especially for more complex filtering conditions.

In [ ]:
df['cylinders'].isin([6,8])

,cylinders
0,True
1,True
2,True
3,True
4,True
...,...
393,False
394,False
395,False
396,False


And then:

In [ ]:
df[df['cylinders'].isin([6,8])].groupby(['origin', 'cylinders']).mean(numeric_only=True)

mpg  displacement  horsepower       weight  \
origin cylinders                                                     
1      6          19.663514    226.283784   99.671233  3213.905405   
       8          14.963107    345.009709  158.300971  4114.718447   
2      6          20.100000    159.750000  113.500000  3382.500000   
3      6          23.883333    156.666667  115.833333  2882.000000   

                  acceleration  model_year  
origin cylinders                            
1      6             16.474324   75.635135  
       8             12.955340   73.902913  
2      6             16.425000   78.250000  
3      6             13.550000   78.000000

<br>

### **Reordering & Sorting MultiIndex**

- How can we swap the index levels so that cylinders comes first?

- How can we sort the index by origin in ascending order and cylinders in descending order?

In [ ]:
# Swap levels
origin_cyl.swaplevel().head()

,,mpg,displacement,horsepower,weight,acceleration,model_year
cylinders,origin,,,,,,
4,1,27.840278,124.284722,80.956522,2437.166667,16.526389,78.027778
6,1,19.663514,226.283784,99.671233,3213.905405,16.474324,75.635135
8,1,14.963107,345.009709,158.300971,4114.718447,12.955340,73.902913
4,2,28.411111,104.222222,78.311475,2330.015873,16.722222,75.507937
5,2,27.366667,145.000000,82.333333,3103.333333,18.633333,79.000000


In [ ]:

# Sort by origin ascending
origin_cyl.sort_index(level='origin', ascending=True)


mpg  displacement  horsepower       weight  \
origin cylinders                                                     
1      4          27.840278    124.284722   80.956522  2437.166667   
       6          19.663514    226.283784   99.671233  3213.905405   
       8          14.963107    345.009709  158.300971  4114.718447   
2      4          28.411111    104.222222   78.311475  2330.015873   
       5          27.366667    145.000000   82.333333  3103.333333   
       6          20.100000    159.750000  113.500000  3382.500000   
3      3          20.550000     72.500000   99.250000  2398.500000   
       4          31.595652     99.768116   75.579710  2153.492754   
       6          23.883333    156.666667  115.833333  2882.000000   

                  acceleration  model_year  
origin cylinders                            
1      4             16.526389   78.027778  
       6             16.474324   75.635135  
       8             12.955340   73.902913  
2      4             16.722222   75.507937  
       5             18.633333   79.000000  
       6             16.425000   78.250000  
3      3             13.250000   75.500000  
       4             16.569565   77.507246  
       6             13.550000   78.000000

In [ ]:

# Sort by cylinders descending
origin_cyl.sort_index(level='cylinders', ascending=False)

,,mpg,displacement,horsepower,weight,acceleration,model_year
origin,cylinders,,,,,,
1,8,14.963107,345.009709,158.300971,4114.718447,12.955340,73.902913
3,6,23.883333,156.666667,115.833333,2882.000000,13.550000,78.000000
2,6,20.100000,159.750000,113.500000,3382.500000,16.425000,78.250000
1,6,19.663514,226.283784,99.671233,3213.905405,16.474324,75.635135
2,5,27.366667,145.000000,82.333333,3103.333333,18.633333,79.000000
3,4,31.595652,99.768116,75.579710,2153.492754,16.569565,77.507246
2,4,28.411111,104.222222,78.311475,2330.015873,16.722222,75.507937
1,4,27.840278,124.284722,80.956522,2437.166667,16.526389,78.027778
3,3,20.550000,72.500000,99.250000,2398.500000,13.250000,75.500000


<br>


#### **Practice Problems**:

**Question:** What is the average fuel efficiency (`mpg`) for cars, grouped by their `model_year` and then by `cylinders`?


In [ ]:
# Solution:

model_year  cylinders
70          4            25.285714
            6            20.500000
            8            14.111111
71          4            27.461538
            6            18.000000
            8            13.428571
72          3            19.000000
            4            23.428571
            8            13.615385
73          3            18.000000
            4            22.727273
            6            19.000000
            8            13.200000
74          4            27.800000
            6            17.857143
            8            14.200000
75          4            25.250000
            6            17.583333
            8            15.666667
76          4            26.766667
            6            20.000000
            8            14.666667
77          3            21.500000
            4            29.107143
            6            19.500000
            8            16.000000
78          4            29.576471
            5            20.300000
            6            19.066667
            8            19.050000
79          4            31.525000
            5            25.400000
            6            22.950000
            8            18.630000
80          3            23.700000
            4            34.612000
            5            36.400000
            6            25.900000
81          4            32.814286
            6            23.428571
            8            26.600000
82          4            32.071429
            6            28.333333
Name: mpg, dtype: float64

<br>

**Question:** What is the maximum `horsepower` for cars, grouped first by their `origin` and then by `model_year`?

In [ ]:
# Solution:

origin  model_year
1       70            225.0
        71            180.0
        72            208.0
        73            230.0
        74            150.0
        75            170.0
        76            180.0
        77            190.0
        78            165.0
        79            155.0
        80            105.0
        81            110.0
        82            112.0
2       70            113.0
        71             90.0
        72            112.0
        73            112.0
        74             83.0
        75            115.0
        76            120.0
        77            110.0
        78            133.0
        79             77.0
        80             88.0
        81             80.0
        82             74.0
3       70             95.0
        71             95.0
        72             97.0
        73            122.0
        74             97.0
        75             97.0
        76            108.0
        77            110.0
        78             97.0
        79             65.0
        80            132.0
        81            120.0
        82             96.0
Name: horsepower, dtype: float64

<br>

**Question:** What is the minimum `acceleration` for cars, grouped first by their `origin` and then by `cylinders`?


In [ ]:
# Solution:

origin  cylinders
1       4            11.6
        6            11.3
        8             8.0
2       4            12.2
        5            15.9
        6            13.6
3       3            12.5
        4            13.5
        6            11.4
Name: acceleration, dtype: float64

<br><br>

## **Advanced Aggregation with agg()**

Sometimes we want different aggregation statistics.

How can you compute multiple aggregate statistics for all numeric columns in the DataFrame?

In [ ]:
import numpy as np
df.select_dtypes(include=np.number).agg(['mean', 'median'])

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
mean,23.514573,5.454774,193.425879,104.469388,2970.424623,15.56809,76.01005,1.572864
median,23.000000,4.000000,148.500000,93.500000,2803.500000,15.50000,76.00000,1.000000


 How can you compute the mean and std only for the MPG and weight columns?

In [ ]:
df.select_dtypes(include=np.number).agg(['mean', 'std'])[['horsepower', 'acceleration']]

,horsepower,acceleration
mean,104.469388,15.568090
std,38.491160,2.757689


How can you apply different aggregation functions to different columns?

In [ ]:
df.select_dtypes(include=np.number).agg({
    'horsepower': ['sum', 'mean'],
    'acceleration': ['mean', 'std']
})

,horsepower,acceleration
sum,40952.000000,NaN
mean,104.469388,15.568090
std,NaN,2.757689




Example: Compute mean and median of MPG, and mean and standard deviation of weight, grouped by origin.

In [ ]:
# Aggregation with groupby + agg
df.groupby('origin').agg({
    'mpg': ['mean', 'median'],
    'weight': ['mean', 'std']
})


mpg              weight            
             mean median         mean         std
origin                                           
1       20.083534   18.5  3361.931727  794.792506
2       27.891429   26.5  2423.300000  490.043191
3       30.450633   31.6  2221.227848  320.497248

<br>


#### **Practice Problems:**

**Question:** For each `origin`, calculate the **minimum**, **maximum**, **mean**, and **median** of the `mpg` (miles per gallon) column.


In [ ]:
# Solution:

,mpg_min,mpg_max,mpg_mean,mpg_median
origin,,,,
1,9.0,39.0,20.083534,18.5
2,16.2,44.3,27.891429,26.5
3,18.0,46.6,30.450633,31.6


<br>

**Question:** For each `cylinders` group, compute the **mean** and **standard deviation** of `horsepower`, and the **count** and **sum** of `weight`.


In [ ]:
# Solution:

horsepower            weight        
                 mean        std  count     sum
cylinders                                      
3           99.250000   8.301606      4    9594
4           78.281407  14.523099    204  470858
5           82.333333  18.583146      3    9310
6          101.506024  14.310472     84  268651
8          158.300971  28.453552    103  423816